# Train Halo Mass Regressor

Train the main PyTorch model for the project: galaxy image -> log10 halo mass regression. The model uses ResNet18 with a scalar regression head and reads settings from `configs/regression_resnet18.yaml`.

## 1. Setup Environment

In [ ]:
# Configure Google Colab environment when needed
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive', force_remount=False)
    project_path = Path('/content/drive/MyDrive/xai-dark-matter-localization')
    os.chdir(project_path)
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))
else:
    project_path = Path.cwd()

print(f'Working directory: {Path.cwd()}')

## 2. Check Dependencies

In [ ]:
# Colab usually includes torch and torchvision. Install only if your runtime is missing a package.
import importlib.util

missing = [pkg for pkg in ['torch', 'torchvision', 'yaml'] if importlib.util.find_spec(pkg) is None]
print('Missing packages:', missing if missing else 'none')

# If needed in Colab, uncomment:
# !pip install -q torch torchvision pyyaml

## 3. Build Regression Metadata If Needed

This step is safe to rerun. It validates images and labels, then writes the CSV used by the PyTorch Dataset.

In [ ]:
from pathlib import Path
from src.data.build_regression_dataset import build_regression_metadata

metadata_csv = Path('data/processed/TNG-DM-XAI-v1/regression_metadata.csv')
source_csv = Path('data/processed/TNG-DM-XAI-v1/metadata_with_images.csv')

if not metadata_csv.exists():
    df_regression = build_regression_metadata(
        input_csv=source_csv,
        output_csv=metadata_csv,
        label_column='group_m_crit200',
        resolution=224,
        project_root=Path.cwd(),
    )
else:
    print(f'Using existing metadata: {metadata_csv}')

## 4. Inspect Config

In [ ]:
from pathlib import Path
import yaml

config_path = Path('configs/regression_resnet18.yaml')
config = yaml.safe_load(config_path.read_text())
config

## 5. Train Model

The training script saves checkpoints, history, predictions, metrics, a loss curve, and a predicted-vs-true scatter plot.

In [ ]:
from src.training.train_regression import train_from_config

metrics = train_from_config(config)
metrics

## 6. Evaluate Checkpoint Again

In [ ]:
from src.training.evaluate_regression import evaluate_from_config

test_metrics = evaluate_from_config(
    config,
    checkpoint_path=Path(config['output']['output_dir']) / 'checkpoints' / 'best_model.pt',
    split='test',
)
test_metrics

## 7. Output Files

In [ ]:
output_dir = Path(config['output']['output_dir'])
for path in sorted(output_dir.rglob('*')):
    if path.is_file():
        print(path)

## 8. Verify Checkpoint Location

Run this after training. It confirms the exact `.pt` checkpoint paths that notebook 8 needs.

In [ ]:
from pathlib import Path

output_dir = Path(config['output']['output_dir'])
expected_checkpoints = [
    output_dir / 'checkpoints' / 'best_model.pt',
    output_dir / 'checkpoints' / 'last_model.pt',
]

print(f'Current working directory: {Path.cwd()}')
for checkpoint in expected_checkpoints:
    print(f'{checkpoint}: exists={checkpoint.exists()}')

if not any(path.exists() for path in expected_checkpoints):
    discovered = sorted(Path.cwd().glob('**/*.pt'))
    print('\nNo expected checkpoint found. Other .pt files under current repo:')
    for path in discovered[:20]:
        print(f'  {path}')
    raise FileNotFoundError(
        'Training did not create best_model.pt or last_model.pt in the configured output directory. '
        'Check that the training cell completed without interruption.'
    )

## Expected output

The notebook trains a ResNet18 halo-mass regressor, saves `best_model.pt`, `last_model.pt`, `training_history.csv`, `predictions_test.csv`, `metrics_test.json`, `loss_curve.png`, and `predicted_vs_true_test.png`. It reports MAE, RMSE, and R2 for the test split.